# 2C · Tarea C: entrada de la agenda referida

Práctica final · **Asistente de agenda en lenguaje natural** · Procesamiento del Lenguaje Natural

Poned en la misma carpeta este notebook, `utilidades.py`, `evaluar.py`, `train.jsonl` y `agenda.jsonl`. Las celdas marcadas con `# TODO` son las que tenéis que completar.

Cada entrada de la agenda es un documento; la frase es la consulta. Cada enfoque devuelve los `id` de la agenda **ordenados de más a menos parecido** (`ref_ranking`). Solo se evalúan las frases con `ref`.

In [ ]:
import time, json
from utilidades import cargar, guardar, particiones, INTENCIONES, TIPOS, HOY
from evaluar import evaluar, imprimir

train, val = particiones()
agenda = cargar("agenda.jsonl")
X_train, X_val = [x["texto"] for x in train], [x["texto"] for x in val]
RESULTADOS = {}      # enfoque -> métricas y predicciones en validación

def registrar(nombre, preds, segundos):
    r = evaluar(val, preds)
    rd = evaluar(val, preds, "dificil") if any(x["dificil"] for x in val) else {}   # hasta que tengáis frases difíciles
    RESULTADOS[nombre] = {"metrica": r["C_acierto@1"], "dificiles": rd.get("C_acierto@1"),
                          "ms_por_frase": 1000 * segundos / max(len(preds), 1), "preds": preds}
    print(f"{nombre:<40} C_acierto@1 = {r['C_acierto@1']:.3f}   difíciles = {rd.get('C_acierto@1', '—')}")

In [ ]:
con_ref = [x for x in val if x["ref"]]
def doc_agenda(a):
    """Texto que representa cada entrada de la agenda. TODO: decidid qué campos incluir."""
    return " ".join(str(v) for v in [a["titulo"], a.get("lugar"), " ".join(a["personas"]), a.get("asignatura")] if v)
DOCS = [doc_agenda(a) for a in agenda]; IDS = [a["id"] for a in agenda]

def registrar_C(nombre, ranking_de):
    t0 = time.time()
    preds = [{"id": x["id"], "ref_ranking": ranking_de(x["texto"])} for x in con_ref]
    registrar(nombre, preds, time.time() - t0)
    print("   MRR =", round(evaluar(val, preds)["C_MRR"], 3))

### C1 · Palabras clave con stemming
Número de raíces en común entre consulta y documento (temas 3 y 5).

In [ ]:
def ranking_C1(consulta):
    return list(IDS)     # TODO: ordenar IDS de más a menos parecido
registrar_C("C1 Palabras clave con stemming", ranking_C1)

### C2 · TF-IDF o BM25
Modelo vectorial o BM25 (`rank_bm25`).

In [ ]:
def ranking_C2(consulta):
    return list(IDS)     # TODO: ordenar IDS de más a menos parecido
registrar_C("C2 TF-IDF o BM25", ranking_C2)

### C3 · Embeddings estáticos
Media de vectores de palabra y similitud coseno.

In [ ]:
def ranking_C3(consulta):
    return list(IDS)     # TODO: ordenar IDS de más a menos parecido
registrar_C("C3 Embeddings estáticos", ranking_C3)

### C4 · Embeddings de frase
sentence-transformers y similitud coseno.

In [ ]:
def ranking_C4(consulta):
    return list(IDS)     # TODO: ordenar IDS de más a menos parecido
registrar_C("C4 Embeddings de frase", ranking_C4)

### C5 · Híbrido
Fusión de los rankings léxico y semántico.

In [ ]:
def ranking_C5(consulta):
    return list(IDS)     # TODO: ordenar IDS de más a menos parecido
registrar_C("C5 Híbrido", ranking_C5)

### C6 · Re-ranking con LLM
Un LLM elige entre los 5 mejores candidatos.

In [ ]:
def ranking_C6(consulta):
    return list(IDS)     # TODO: ordenar IDS de más a menos parecido
registrar_C("C6 Re-ranking con LLM", ranking_C6)

## Resumen y enfoque elegido
Guardamos los resultados de validación para el notebook 3 y elegimos un enfoque.

In [ ]:
import pandas as pd
print(pd.DataFrame({k: {c: v[c] for c in ("metrica", "dificiles", "ms_por_frase")} for k, v in RESULTADOS.items()}).T.round(3))
json.dump(RESULTADOS, open("resultados_C.json", "w", encoding="utf-8"), ensure_ascii=False)
MEJOR = max(RESULTADOS, key=lambda k: RESULTADOS[k]["metrica"])     # TODO: elegid y justificad el enfoque (calidad, coste, latencia…)
print("Enfoque elegido:", MEJOR)

## Exportar el enfoque elegido
Copiad a **`tarea_c.py`** el enfoque elegido (ya entrenado o con lo necesario para entrenarlo) como una función `ranking_agenda(texto, agenda) -> ids de la agenda, de más a menos parecido`. La usan el asistente (notebook 4) y la función `generar_predicciones` del notebook 3, que es la que ejecutará el profesor con el **test oculto**.